In [ ]:
import os
import torch
import random
import re
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords

In [ ]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [ ]:
# Make stop words.
nltk.download('stopwords')
stop_words = list(stopwords.words('english'))
stop_words.extend(['mrs', 'ms', 'mr', 'am', 'pm'])
# Map dataset labels to 1s or 0s.
label_map = {
    "met": 0,
    "unmet": 1
}
# Load the dataset
dataset = pd.read_csv("./dataSyntheticAll.csv")
dataset["label"] = dataset["needs"].map(label_map)

# Make function to remove punctuation, make lowercase, remove names, remove stopwords, punctuation, lemmatize, remove documents with less than 5 tokens.
def preprocessing(notes, min_words=1):
    cleaned_notes = []

    for note in notes:
        # lowercase + extract words only
        tokens = re.findall(r"\b[a-zA-Z]+\b", note.lower())
        
        # filter stopwords
        tokens = [t for t in tokens if t not in stop_words]
        
        if len(tokens) >= min_words:
            cleaned_notes.append(" ".join(tokens))

    return cleaned_notes

dataset['report'] = preprocessing(dataset['report'].values.tolist())

# Check distributions.
def check_distribution(dataframe, name):
    counts = dataframe["label"].value_counts(normalize=True)
    print(f"{name} distribution:")
    print(counts)

check_distribution(dataset, "All Data")